In [ ]:
import cv2
import numpy as np
import random
import math
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
import pandas as pd
from scipy.ndimage import convolve

In [ ]:
def embed(er, img):
    output = []
    h, w, channel = img.shape
    mean_list = [np.mean(img[:, :, c]) for c in range(3)]
    mean_gray = sum(mean_list) / 3

    for c in range(channel):
        pixel = img[:, :, c].astype(np.int16)
        amount_of_data = math.ceil((h*w*er)) if math.ceil((h*w*er)) > 16 else 16
        random_length = math.ceil(amount_of_data/w) * w
        data = pixel.flatten()[:random_length].reshape(-1, pixel.shape[1])
        remaining_data = pixel.flatten()[amount_of_data:]
        pixel_float = data.astype(float)

        b = mean_gray / mean_list[c] * pixel_float
        a = np.where(b > 255, 255 - 1e-5, b)

        if random_length > a.size:
            m_array = np.random.randint(0, 2, size=a.shape)
        else:
            m_array = np.zeros(a.shape)

            if (random_length < h*w):
                random_length = random_length-16 if random_length-16 > 0 else 0

            random_values = np.random.randint(0, 2, size=random_length)
            m_array.flat[:random_length] = random_values

        floor_a = np.floor(a)
        ceil_a = np.ceil(a)
        processed_img = np.zeros_like(a)
        condition1 = (floor_a % 2) == m_array
        condition2 = (ceil_a % 2) == m_array
        processed_img = np.where(condition1, floor_a, processed_img)
        processed_img = np.where((processed_img == 0) & condition2, ceil_a, processed_img)
        processed_img = np.concatenate((processed_img.flatten()[:amount_of_data], remaining_data), axis=0).reshape(-1, pixel.shape[1])
        processed_img = np.clip(processed_img, 0, 255).astype(np.uint8)
        output.append(processed_img)

    stego_image = cv2.merge(output)
    return stego_image

In [ ]:
folder_path = "/content/drive/MyDrive/dataset/inria holidays/color"
result_path = f"/content/drive/MyDrive/result/inria holidays/color/proposed method 2/"

In [ ]:
results = []
ER = [0.5, 1, 1.5, 2, 2.5, 3]

for embedding_rate in ER:
    er = embedding_rate / 3
    for experiment in range(1, 11):
        new_folder = f"{result_path}/{embedding_rate}/{experiment}"
        file_count = 1
        files = os.listdir(folder_path)

        for filename in files:
            cover_path = os.path.join(folder_path, filename)
            cover = cv2.imread(cover_path)

            output = embed(er, cover)

            stego_path = f"{new_folder}/{filename}"
            print(f"{stego_path}")
            cv2.imwrite(stego_path, output)

            if(file_count % 100 == 0):
                print(f"Embedding rate : {embedding_rate} # Experiment : {experiment} # Progress : {round((file_count / len(files)) * 100, 2)}%")

            file_count += 1

print(f"Done!")